<a href="https://colab.research.google.com/github/s34836/HTW_DS/blob/main/Exercise_Sheet_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Statistical Learning - Exercise Sheet 2

Notebook solution for Exercises 1--5.

Assumption: the required CSV files are located in the root/current working directory:

- `BostonHousing.csv`
- `weekly.csv`

The notebook is written so that each exercise can be run independently after the import/setup cells.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    accuracy_score,
    f1_score,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.datasets import load_iris

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RANDOM_STATE = 42

## Exercise 1: Gradient Descent Method

We estimate the parameters of simple linear regression

\[
y_i = \beta_0 + \beta_1 x_i + \varepsilon_i
\]

by minimizing

\[
J(\beta)=\frac{1}{n}\sum_{i=1}^{n}(y_i-\beta_0-\beta_1x_i)^2.
\]

In [ ]:
# Example data for demonstrating gradient descent
np.random.seed(RANDOM_STATE)

n = 100
x = np.linspace(0, 10, n)
y = 3.0 + 2.0 * x + np.random.normal(0, 2, size=n)

plt.figure(figsize=(7, 5))
plt.scatter(x, y)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Example data for simple linear regression")
plt.show()

In [ ]:
def mse_loss(beta_0, beta_1, x, y):
    y_hat = beta_0 + beta_1 * x
    return np.mean((y - y_hat) ** 2)


def gradient_descent_simple_lr(x, y, alpha=0.01, n_iter=5000):
    beta_0 = 0.0
    beta_1 = 0.0
    n = len(y)
    history = []

    for t in range(n_iter):
        residuals = y - beta_0 - beta_1 * x
        grad_beta_0 = -(2 / n) * np.sum(residuals)
        grad_beta_1 = -(2 / n) * np.sum(x * residuals)

        beta_0 -= alpha * grad_beta_0
        beta_1 -= alpha * grad_beta_1

        if t % 50 == 0:
            history.append(mse_loss(beta_0, beta_1, x, y))

    return beta_0, beta_1, history

beta_0_gd, beta_1_gd, loss_history = gradient_descent_simple_lr(x, y, alpha=0.01, n_iter=5000)

print(f"Estimated beta_0: {beta_0_gd:.4f}")
print(f"Estimated beta_1: {beta_1_gd:.4f}")
print(f"Final MSE: {mse_loss(beta_0_gd, beta_1_gd, x, y):.4f}")

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(loss_history)
plt.xlabel("Saved iteration index")
plt.ylabel("MSE loss")
plt.title("Gradient descent convergence")
plt.show()

plt.figure(figsize=(7, 5))
plt.scatter(x, y, label="Observed data")
plt.plot(x, beta_0_gd + beta_1_gd * x, label="Gradient descent fit")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Simple linear regression fitted by gradient descent")
plt.legend()
plt.show()

## Exercise 2: Linear Regression -- Regularization

The goal is to predict the crime rate `CRIM` from selected Boston Housing variables. The CSV file is assumed to be available as `BostonHousing.csv` in the current working directory.

In [ ]:
boston = pd.read_csv("BostonHousing.csv")

print("Shape:", boston.shape)
display(boston.head())
display(boston.describe())
boston.info()

In [ ]:
# Normalize column names to uppercase for easier handling if the CSV uses lowercase names
boston.columns = [c.upper() for c in boston.columns]

print(boston.columns.tolist())
print("\nMissing values:")
print(boston.isna().sum())

In [ ]:
# Scatterplots against CRIM
candidate_features = ["RM", "LSTAT", "DIS", "TAX", "NOX", "INDUS", "AGE", "PTRATIO"]
features_available = [c for c in candidate_features if c in boston.columns]

for col in features_available:
    plt.figure(figsize=(6, 4))
    plt.scatter(boston[col], boston["CRIM"], alpha=0.7)
    plt.xlabel(col)
    plt.ylabel("CRIM")
    plt.title(f"CRIM vs {col}")
    plt.show()

In [ ]:
# Correlation with CRIM
corr_with_crim = boston.corr(numeric_only=True)["CRIM"].sort_values(ascending=False)
display(corr_with_crim.to_frame("correlation_with_CRIM"))

In [ ]:
# Multivariate linear regression and lasso regression
selected_features = [c for c in ["RM", "LSTAT", "DIS", "TAX", "NOX"] if c in boston.columns]

X = boston[selected_features]
y = boston["CRIM"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

lasso_model = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", Lasso(alpha=0.1, random_state=RANDOM_STATE, max_iter=10000))
])
lasso_model.fit(X_train, y_train)

models = {
    "Linear Regression": linear_model,
    "Lasso Regression": lasso_model,
}

for name, model in models.items():
    pred = model.predict(X_test)
    print(name)
    print("MAE:", round(mean_absolute_error(y_test, pred), 4))
    print("RMSE:", round(np.sqrt(mean_squared_error(y_test, pred)), 4))
    print("R2:", round(r2_score(y_test, pred), 4))
    print()

In [ ]:
# Coefficient comparison
linear_coef = pd.Series(linear_model.coef_, index=selected_features, name="linear_regression")
lasso_coef = pd.Series(
    lasso_model.named_steps["lasso"].coef_,
    index=selected_features,
    name="lasso_scaled_features"
)

coef_table = pd.concat([linear_coef, lasso_coef], axis=1)
display(coef_table)

## Exercise 3: Subset Selection

Subset selection searches for a smaller subset of predictors that explains the response variable well. Below is a simple forward selection example for the Boston Housing data. The criterion used here is test-set RMSE.

In [ ]:
def evaluate_feature_subset(data, features, target="CRIM"):
    X = data[list(features)]
    y = data[target]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )
    model = LinearRegression()
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    return rmse

numeric_features = [c for c in boston.select_dtypes(include=[np.number]).columns if c != "CRIM"]

remaining = numeric_features.copy()
selected = []
steps = []

while remaining:
    candidates = []
    for feature in remaining:
        candidate_features = selected + [feature]
        rmse = evaluate_feature_subset(boston, candidate_features, target="CRIM")
        candidates.append((feature, rmse))

    best_feature, best_rmse = min(candidates, key=lambda x: x[1])
    selected.append(best_feature)
    remaining.remove(best_feature)
    steps.append({
        "step": len(selected),
        "added_feature": best_feature,
        "features": selected.copy(),
        "test_rmse": best_rmse,
    })

forward_selection_results = pd.DataFrame(steps)
display(forward_selection_results[["step", "added_feature", "test_rmse"]])

In [ ]:
best_step = forward_selection_results.loc[forward_selection_results["test_rmse"].idxmin()]
print("Best step:", int(best_step["step"]))
print("Best feature subset:", best_step["features"])
print("Best test RMSE:", round(best_step["test_rmse"], 4))

## Exercise 4: Logistic Regression

The `Weekly` data set contains weekly stock-market returns and a binary target variable `Direction`, where the market went either `UP` or `DOWN`. The CSV file is assumed to be available as `weekly.csv` in the current working directory.

In [ ]:
weekly = pd.read_csv("weekly.csv")

print("Shape:", weekly.shape)
display(weekly.head())
display(weekly.describe())
weekly.info()

In [ ]:
# Normalize column names in case of minor capitalization differences
weekly.columns = [c.strip() for c in weekly.columns]

print(weekly.columns.tolist())
print("\nDirection counts:")
print(weekly["Direction"].value_counts())

In [ ]:
# Pairwise scatterplots for selected numeric variables
scatter_vars = ["Lag1", "Lag2", "Lag3", "Lag4", "Lag5", "Today", "Volume"]
scatter_vars = [c for c in scatter_vars if c in weekly.columns]

pd.plotting.scatter_matrix(
    weekly[scatter_vars],
    figsize=(12, 12),
    diagonal="hist",
    alpha=0.7
)
plt.suptitle("Pairwise scatterplots for Weekly data", y=1.02)
plt.show()

In [ ]:
# Logistic regression using all lag variables
lag_features = ["Lag1", "Lag2", "Lag3", "Lag4", "Lag5"]

X = weekly[lag_features]
y = weekly["Direction"]

logit_all = LogisticRegression(max_iter=1000)
logit_all.fit(X, y)

coef_all = pd.Series(logit_all.coef_[0], index=lag_features, name="coefficient")
print("Intercept:", logit_all.intercept_[0])
display(coef_all.to_frame())

In [ ]:
# Train-test split: train up to 2008, test from 2009 onward
train_mask = weekly["Year"] <= 2008
test_mask = weekly["Year"] >= 2009

X_train = weekly.loc[train_mask, ["Lag1"]]
y_train = weekly.loc[train_mask, "Direction"]
X_test = weekly.loc[test_mask, ["Lag1"]]
y_test = weekly.loc[test_mask, "Direction"]

logit_lag1 = LogisticRegression(max_iter=1000)
logit_lag1.fit(X_train, y_train)

print("Classes:", logit_lag1.classes_)
print("Intercept:", logit_lag1.intercept_[0])
print("Coefficient for Lag1:", logit_lag1.coef_[0][0])

print("\nFitted model:")
print(f"logit(P(Direction = {logit_lag1.classes_[1]})) = {logit_lag1.intercept_[0]:.4f} + {logit_lag1.coef_[0][0]:.4f} * Lag1")

In [ ]:
y_pred = logit_lag1.predict(X_test)

cm = confusion_matrix(y_test, y_pred, labels=logit_lag1.classes_)
print(cm)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification report:")
print(classification_report(y_test, y_pred))

ConfusionMatrixDisplay(cm, display_labels=logit_lag1.classes_).plot()
plt.title("Confusion matrix: logistic regression with Lag1")
plt.show()

## Exercise 5: Model Evaluation -- Classification

This exercise investigates ROC curves, precision-recall curves, F-score, and class imbalance.

In [ ]:
# Helper functions for binary classification evaluation

def evaluate_binary_classifier(y_true, y_score, threshold=0.5, positive_label=1, title="Model"):
    y_pred = (y_score >= threshold).astype(int)

    print(title)
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("F1-score:", round(f1_score(y_true, y_pred, pos_label=positive_label), 4))
    print("ROC-AUC:", round(roc_auc_score(y_true, y_score), 4))
    print("Average precision:", round(average_precision_score(y_true, y_score), 4))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))


def plot_roc_and_pr(y_true, y_score, title_prefix="Model"):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    precision, recall, _ = precision_recall_curve(y_true, y_score)

    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc_score(y_true, y_score):.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{title_prefix}: ROC curve")
    plt.legend()
    plt.show()

    plt.figure(figsize=(7, 5))
    plt.plot(recall, precision, label=f"Average precision = {average_precision_score(y_true, y_score):.3f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{title_prefix}: Precision-Recall curve")
    plt.legend()
    plt.show()

In [ ]:
# Load Iris and reduce it to binary classification: setosa vs versicolor
iris = load_iris(as_frame=True)
iris_df = iris.frame.copy()
iris_df["species"] = iris_df["target"].map(dict(enumerate(iris.target_names)))

binary_iris = iris_df[iris_df["species"].isin(["setosa", "versicolor"])].copy()
binary_iris["target_binary"] = (binary_iris["species"] == "versicolor").astype(int)

print(binary_iris["species"].value_counts())
display(binary_iris.head())

In [ ]:
# Balanced binary Iris data set
X_balanced = binary_iris[iris.feature_names]
y_balanced = binary_iris["target_binary"]

print("Balanced class counts:")
print(y_balanced.value_counts())

In [ ]:
# Artificially unbalanced data set:
# keep all setosa observations but only a small number of versicolor observations
positive = binary_iris[binary_iris["target_binary"] == 1]
negative = binary_iris[binary_iris["target_binary"] == 0]

positive_small = positive.sample(n=10, random_state=RANDOM_STATE)
unbalanced_iris = pd.concat([negative, positive_small], axis=0).sample(frac=1, random_state=RANDOM_STATE)

X_unbalanced = unbalanced_iris[iris.feature_names]
y_unbalanced = unbalanced_iris["target_binary"]

print("Unbalanced class counts:")
print(y_unbalanced.value_counts())

In [ ]:
# Train and evaluate logistic regression on balanced data
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_balanced, y_balanced,
    test_size=0.3,
    random_state=RANDOM_STATE,
    stratify=y_balanced
)

clf_balanced = Pipeline([
    ("scaler", StandardScaler()),
    ("logit", LogisticRegression(max_iter=1000))
])
clf_balanced.fit(X_train_b, y_train_b)

score_balanced = clf_balanced.predict_proba(X_test_b)[:, 1]
evaluate_binary_classifier(y_test_b, score_balanced, title="Balanced Iris classifier")
plot_roc_and_pr(y_test_b, score_balanced, title_prefix="Balanced Iris")

In [ ]:
# Train and evaluate logistic regression on unbalanced data
X_train_u, X_test_u, y_train_u, y_test_u = train_test_split(
    X_unbalanced, y_unbalanced,
    test_size=0.3,
    random_state=RANDOM_STATE,
    stratify=y_unbalanced
)

clf_unbalanced = Pipeline([
    ("scaler", StandardScaler()),
    ("logit", LogisticRegression(max_iter=1000))
])
clf_unbalanced.fit(X_train_u, y_train_u)

score_unbalanced = clf_unbalanced.predict_proba(X_test_u)[:, 1]
evaluate_binary_classifier(y_test_u, score_unbalanced, title="Unbalanced Iris classifier")
plot_roc_and_pr(y_test_u, score_unbalanced, title_prefix="Unbalanced Iris")

In [ ]:
# Direct comparison of F1-score, ROC-AUC, and average precision
comparison = pd.DataFrame({
    "dataset": ["balanced", "unbalanced"],
    "positive_fraction": [y_balanced.mean(), y_unbalanced.mean()],
    "f1_score": [
        f1_score(y_test_b, (score_balanced >= 0.5).astype(int)),
        f1_score(y_test_u, (score_unbalanced >= 0.5).astype(int)),
    ],
    "roc_auc": [
        roc_auc_score(y_test_b, score_balanced),
        roc_auc_score(y_test_u, score_unbalanced),
    ],
    "average_precision": [
        average_precision_score(y_test_b, score_balanced),
        average_precision_score(y_test_u, score_unbalanced),
    ],
})

display(comparison)

## Final notes

- ROC curves are usually less sensitive to class imbalance because both TPR and FPR are normalized by the number of positives and negatives separately.
- Precision-recall curves are more sensitive to class imbalance because precision directly depends on false positives.
- F1-score is also sensitive to class imbalance because it combines precision and recall.